# Fixed: Hierarchical Multi-Label Classification with LLM Labels
## Critical Fix: Prevents empty labels during pseudo-label updates
## Improved Version: Enhanced Silver Labeling, GNN, and Self-Training

In [1]:
import os
import random
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Path Configuration
ROOT = r"F:/work/BIGdata/project_release/Amazon_products"
TRAIN_PATH = os.path.join(ROOT, "train/train_corpus.txt")
TEST_PATH = os.path.join(ROOT, "test/test_corpus.txt")
HIERARCHY_PATH = os.path.join(ROOT, "class_hierarchy.txt")
KEYWORDS_PATH = os.path.join(ROOT, "class_related_keywords.txt")
MODEL_SAVE_DIR = r'F:/work/BIGdata/project_release/model'

# LLM generated data paths
LLM_DATA_PATH = os.path.join(ROOT, "llm_generated_data.pkl")
FINAL_LABELS_PATH = os.path.join(MODEL_SAVE_DIR, "final_train_labels.pkl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# --- Data Loading ---
def load_data_and_graph():
    documents = []
    with open(TRAIN_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                documents.append({'id': int(parts[0]), 'text': parts[1]})
    
    test_documents = []
    if os.path.exists(TEST_PATH):
        with open(TEST_PATH, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split('\t', 1)
                if len(parts) == 2:
                    test_documents.append({'id': int(parts[0]), 'text': parts[1]})
    
    class_info = {}
    with open(KEYWORDS_PATH, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            parts = line.strip().split(':')
            if len(parts) == 2:
                name = parts[0].strip().replace('_', ' ')
                keywords = parts[1].strip()
                class_info[idx] = {'name': name, 'keywords': keywords}
    
    G = nx.DiGraph()
    with open(HIERARCHY_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            p, c = map(int, line.strip().split())
            G.add_edge(p, c)
    
    num_classes = len(class_info)
    roots = [n for n, d in G.in_degree() if d == 0]
    
    return documents, test_documents, G, class_info, num_classes, roots

all_docs, test_docs, G, class_info, num_classes, roots = load_data_and_graph()
print(f"Loaded {len(all_docs)} training docs, {len(test_docs)} test docs, and {num_classes} classes")

In [ ]:
# --- Load Pre-generated LLM Labels ---

def load_llm_labels():
    val_data = None
    seed_train_labels = None
    unlabeled_indices = None
    final_train_labels = None
    
    if os.path.exists(LLM_DATA_PATH):
        print(f"Loading LLM labels from {LLM_DATA_PATH}...")
        with open(LLM_DATA_PATH, 'rb') as f:
            data = pickle.load(f)
            val_data = data.get('val_data', [])
            seed_train_labels = data.get('seed_train_labels', {})
            unlabeled_indices = data.get('unlabeled_indices', [])
        
        print(f"  - Validation data: {len(val_data)} samples")
        print(f"  - Seed train labels: {len(seed_train_labels)} samples")
        print(f"  - Unlabeled indices: {len(unlabeled_indices)} samples")
    
    if os.path.exists(FINAL_LABELS_PATH):
        print(f"\nLoading final train labels from {FINAL_LABELS_PATH}...")
        with open(FINAL_LABELS_PATH, 'rb') as f:
            final_train_labels = pickle.load(f)
        print(f"  - Final train labels: {len(final_train_labels)} samples")
    
    return val_data, seed_train_labels, unlabeled_indices, final_train_labels

val_data, seed_train_labels, unlabeled_indices, final_train_labels = load_llm_labels()

if final_train_labels is not None:
    print("\n==> Using final_train_labels for training")
    use_final_labels = True
elif seed_train_labels is not None:
    print("\n==> Using seed_train_labels for training")
    use_final_labels = False
else:
    raise ValueError("No pre-generated labels found!")

In [ ]:
# --- Hierarchical Constraint Functions ---

def get_ancestors(G, node):
    ancestors = set()
    queue = [node]
    while queue:
        current = queue.pop(0)
        for parent in G.predecessors(current):
            if parent not in ancestors:
                ancestors.add(parent)
                queue.append(parent)
    return ancestors

def get_descendants(G, node):
    """IMPROVED: Get all descendant nodes"""
    descendants = set()
    queue = [node]
    while queue:
        current = queue.pop(0)
        for child in G.successors(current):
            if child not in descendants:
                descendants.add(child)
                queue.append(child)
    return descendants

def get_depth(G, node, roots):
    """IMPROVED: Calculate node depth in hierarchy"""
    if node in roots:
        return 0
    ancestors = get_ancestors(G, node)
    if not ancestors:
        return 0
    return max(get_depth(G, a, roots) for a in G.predecessors(node)) + 1

def enforce_hierarchy_constraint(labels, G):
    constrained_labels = set(labels)
    for label in labels:
        ancestors = get_ancestors(G, label)
        constrained_labels.update(ancestors)
    return list(constrained_labels)

# IMPROVED: Precompute hierarchy information
print("\nPrecomputing hierarchy information...")
node_depths = {}
for node in G.nodes():
    node_depths[node] = get_depth(G, node, roots)
max_depth = max(node_depths.values()) if node_depths else 0
print(f"Max hierarchy depth: {max_depth}")

# Get leaf nodes (most specific classes)
leaf_nodes = set([n for n in G.nodes() if G.out_degree(n) == 0])
print(f"Number of leaf nodes: {len(leaf_nodes)}")

# Apply hierarchy constraint
print("\nApplying hierarchy constraints...")

if use_final_labels:
    for doc_id in final_train_labels:
        original = final_train_labels[doc_id]
        constrained = enforce_hierarchy_constraint(original, G)
        final_train_labels[doc_id] = constrained
    print(f"Applied to {len(final_train_labels)} final labels")
else:
    for idx in seed_train_labels:
        original = seed_train_labels[idx]
        constrained = enforce_hierarchy_constraint(original, G)
        seed_train_labels[idx] = constrained
    print(f"Applied to {len(seed_train_labels)} seed labels")

if val_data:
    for item in val_data:
        original = item['labels']
        constrained = enforce_hierarchy_constraint(original, G)
        item['labels'] = constrained
    print(f"Applied to {len(val_data)} validation labels")

In [ ]:
# --- IMPROVED Silver Labeling with Hybrid Approach ---

def generate_silver_labels_hybrid(documents, unlabeled_indices, class_info, num_classes, 
                                   seed_labels=None, G=None):
    """
    IMPROVED: Hybrid silver labeling using:
    1. TF-IDF similarity
    2. Keyword matching with hierarchy awareness
    3. Label propagation from seed labels
    """
    print("\n=== Generating Improved Silver Labels ===")
    
    # 1. TF-IDF based similarity
    print("Step 1: Computing TF-IDF similarity...")
    class_descriptions = []
    for i in range(num_classes):
        # Include name multiple times for emphasis
        desc = f"{class_info[i]['name']} {class_info[i]['name']} {class_info[i]['keywords']}"
        class_descriptions.append(desc)
    
    unlabeled_docs = [documents[i] for i in unlabeled_indices]
    doc_texts = [d['text'] for d in unlabeled_docs]
    
    # Use improved TF-IDF settings
    vectorizer = TfidfVectorizer(
        max_features=8000,  # Increased
        stop_words='english', 
        ngram_range=(1, 3),  # Extended to trigrams
        min_df=2,
        max_df=0.95,
        sublinear_tf=True  # Apply sublinear scaling
    )
    
    all_texts = doc_texts + class_descriptions
    vectorizer.fit(all_texts)
    
    doc_vectors = vectorizer.transform(doc_texts)
    class_vectors = vectorizer.transform(class_descriptions)
    tfidf_similarity = cosine_similarity(doc_vectors, class_vectors)
    
    # 2. Keyword matching score
    print("Step 2: Computing keyword matching scores...")
    keyword_scores = np.zeros((len(unlabeled_indices), num_classes))
    
    for doc_idx, text in enumerate(doc_texts):
        text_lower = text.lower()
        for class_id in range(num_classes):
            keywords = class_info[class_id]['keywords'].lower().split(',')
            name_words = class_info[class_id]['name'].lower().split()
            
            # Count keyword matches
            match_count = 0
            for kw in keywords + name_words:
                kw = kw.strip()
                if kw and kw in text_lower:
                    match_count += 1
            
            # Normalize by number of keywords
            total_kw = len(keywords) + len(name_words)
            if total_kw > 0:
                keyword_scores[doc_idx, class_id] = match_count / total_kw
    
    # 3. Hierarchy-aware scoring
    print("Step 3: Applying hierarchy-aware scoring...")
    # Boost scores for more specific (deeper) classes
    depth_weights = np.zeros(num_classes)
    for class_id in range(num_classes):
        depth = node_depths.get(class_id, 0)
        # Give higher weight to deeper (more specific) classes
        depth_weights[class_id] = 1.0 + (depth / (max_depth + 1)) * 0.5
    
    # 4. Combine scores
    print("Step 4: Combining scores...")
    # Weighted combination
    combined_scores = (
        0.6 * tfidf_similarity + 
        0.3 * keyword_scores +
        0.1 * (tfidf_similarity * depth_weights)  # Hierarchy bonus
    )
    
    # 5. Label propagation from seed labels (if available)
    if seed_labels:
        print("Step 5: Applying label propagation...")
        # Build document similarity for propagation
        seed_indices = list(seed_labels.keys())
        seed_texts = [documents[i]['text'] for i in seed_indices]
        
        seed_vectors = vectorizer.transform(seed_texts)
        doc_to_seed_sim = cosine_similarity(doc_vectors, seed_vectors)
        
        # Propagate labels from similar seed documents
        propagated_scores = np.zeros((len(unlabeled_indices), num_classes))
        for doc_idx in range(len(unlabeled_indices)):
            # Find top-k most similar seed documents
            top_k = 5
            top_seed_indices = np.argsort(doc_to_seed_sim[doc_idx])[-top_k:]
            
            for seed_idx in top_seed_indices:
                seed_doc_idx = seed_indices[seed_idx]
                sim_score = doc_to_seed_sim[doc_idx, seed_idx]
                
                if sim_score > 0.3:  # Only propagate from similar documents
                    for label in seed_labels[seed_doc_idx]:
                        if label < num_classes:
                            propagated_scores[doc_idx, label] += sim_score
        
        # Normalize propagated scores
        max_prop = propagated_scores.max()
        if max_prop > 0:
            propagated_scores /= max_prop
        
        # Add to combined scores
        combined_scores = 0.7 * combined_scores + 0.3 * propagated_scores
    
    # 6. Generate final silver labels
    print("Step 6: Generating final silver labels...")
    silver_labels = {}
    
    for doc_idx, unlabeled_idx in enumerate(tqdm(unlabeled_indices, desc="Assigning labels")):
        scores = combined_scores[doc_idx]
        
        # Dynamic threshold based on score distribution
        threshold = max(np.percentile(scores, 90), 0.15)
        
        # Get candidates above threshold
        candidates = np.where(scores > threshold)[0].tolist()
        
        # If too few candidates, take top-k
        if len(candidates) < 2:
            candidates = np.argsort(scores)[-3:].tolist()
        
        # Prioritize leaf nodes (most specific classes)
        leaf_candidates = [c for c in candidates if c in leaf_nodes]
        non_leaf_candidates = [c for c in candidates if c not in leaf_nodes]
        
        # Select labels: prefer specific classes
        if len(leaf_candidates) >= 1:
            # Sort by score and take top
            leaf_candidates.sort(key=lambda x: scores[x], reverse=True)
            selected = leaf_candidates[:2]
        else:
            selected = sorted(candidates, key=lambda x: scores[x], reverse=True)[:2]
        
        # Apply hierarchy constraint
        selected = enforce_hierarchy_constraint(selected, G)
        
        # Limit to top 3 after hierarchy expansion
        if len(selected) > 3:
            selected_scores = [(c, scores[c]) for c in selected]
            selected_scores.sort(key=lambda x: x[1], reverse=True)
            selected = [c for c, _ in selected_scores[:3]]
        
        silver_labels[unlabeled_idx] = selected
    
    # Statistics
    avg_labels = np.mean([len(v) for v in silver_labels.values()])
    leaf_coverage = sum(1 for v in silver_labels.values() if any(l in leaf_nodes for l in v)) / len(silver_labels)
    print(f"\nSilver label statistics:")
    print(f"  - Average labels per doc: {avg_labels:.2f}")
    print(f"  - Documents with leaf labels: {leaf_coverage*100:.1f}%")
    
    return silver_labels

if not use_final_labels and unlabeled_indices:
    silver_labels = generate_silver_labels_hybrid(
        all_docs, unlabeled_indices, class_info, num_classes,
        seed_labels=seed_train_labels, G=G
    )
    print(f"Generated silver labels for {len(silver_labels)} documents")
    combined_train_labels = {**seed_train_labels, **silver_labels}
    print(f"Total training labels: {len(combined_train_labels)}")
else:
    combined_train_labels = final_train_labels
    print(f"Using {len(combined_train_labels)} final training labels")

In [ ]:
# --- IMPROVED Build Adjacency Matrix with Label Correlation ---

def build_adjacency_matrix_improved(G, num_classes, train_labels=None):
    """
    IMPROVED: Build adjacency matrix with:
    1. Hierarchy edges
    2. Label co-occurrence correlation
    """
    # Base: Identity + Hierarchy edges
    A = torch.eye(num_classes)
    for u, v in G.edges():
        A[u, v] = 1
        A[v, u] = 1
    
    # Add label co-occurrence edges from training data
    if train_labels:
        cooccurrence = np.zeros((num_classes, num_classes))
        for labels in train_labels.values():
            for i, l1 in enumerate(labels):
                for l2 in labels[i+1:]:
                    if l1 < num_classes and l2 < num_classes:
                        cooccurrence[l1, l2] += 1
                        cooccurrence[l2, l1] += 1
        
        # Normalize co-occurrence
        row_sums = cooccurrence.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        cooccurrence_norm = cooccurrence / row_sums
        
        # Add strong co-occurrence edges (threshold: 0.1)
        cooccurrence_edges = torch.tensor(cooccurrence_norm > 0.1, dtype=torch.float32)
        A = A + 0.5 * cooccurrence_edges  # Weighted addition
        A = torch.clamp(A, 0, 1)  # Ensure max is 1
    
    # Symmetric normalization
    degree = A.sum(dim=1)
    degree_inv_sqrt = torch.pow(degree, -0.5)
    degree_inv_sqrt[torch.isinf(degree_inv_sqrt)] = 0
    D_inv_sqrt = torch.diag(degree_inv_sqrt)
    A_hat = D_inv_sqrt @ A @ D_inv_sqrt
    
    return A_hat

A_hat = build_adjacency_matrix_improved(G, num_classes, combined_train_labels).to(device)
print(f"Adjacency matrix shape: {A_hat.shape}")

In [ ]:
# --- Dataset ---

class TextDataset(Dataset):
    def __init__(self, texts, labels_dict, tokenizer, num_classes, max_length=128):
        self.texts = texts
        self.labels_dict = labels_dict
        self.tokenizer = tokenizer
        self.num_classes = num_classes
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        
        label_vector = torch.zeros(self.num_classes)
        if idx in self.labels_dict:
            for label in self.labels_dict[idx]:
                if label < self.num_classes:
                    label_vector[label] = 1
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': label_vector
        }

In [ ]:
# --- IMPROVED Model with Hierarchical Attention ---

class HierarchicalGCNClassifier(nn.Module):
    """
    IMPROVED model with:
    1. Multiple GCN layers with residual connections @
    2. Hierarchical attention mechanism
    3. Label-wise attention
    4. Dropout regularization
    """
    def __init__(self, num_classes, bert_model='bert-base-uncased', hidden_dim=768, gcn_layers=3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model)
        self.num_classes = num_classes
        self.hidden_dim = hidden_dim
        
        # IMPROVED: More GCN layers with residual connections
        self.gcn_layers = nn.ModuleList([
            nn.Linear(hidden_dim, hidden_dim) for _ in range(gcn_layers)
        ])
        self.gcn_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(gcn_layers)
        ])
        
        # Learnable class embeddings
        self.class_embeddings = nn.Parameter(torch.randn(num_classes, hidden_dim) * 0.02)
        
        # Multi-head attention for document-class interaction
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=8, batch_first=True, dropout=0.1)
        
        # IMPROVED: Label-wise attention
        self.label_attention = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # IMPROVED: Deeper classifier with residual
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, input_ids, attention_mask, A_hat):
        # BERT encoding
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use mean pooling instead of just [CLS] token
        hidden_states = bert_output.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_hidden = torch.sum(hidden_states * mask_expanded, dim=1)
        sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
        doc_embed = sum_hidden / sum_mask
        
        # GCN for class embeddings with residual connections
        class_feats = self.class_embeddings
        for gcn_layer, gcn_norm in zip(self.gcn_layers, self.gcn_norms):
            residual = class_feats
            class_feats = gcn_layer(A_hat @ class_feats)
            class_feats = F.gelu(class_feats)
            class_feats = gcn_norm(class_feats)
            class_feats = class_feats + residual  # Residual connection
        
        class_feats = self.dropout(class_feats)
        
        # Document-class attention
        doc_expanded = doc_embed.unsqueeze(1)
        class_expanded = class_feats.unsqueeze(0).expand(doc_embed.size(0), -1, -1)
        
        attn_output, attn_weights = self.attention(doc_expanded, class_expanded, class_expanded)
        attn_output = attn_output.squeeze(1)
        
        # Combine document and attention features
        combined = torch.cat([doc_embed, attn_output], dim=1)
        logits = self.classifier(combined)
        
        return logits

In [ ]:
# --- IMPROVED Loss Functions ---

class AsymmetricLoss(nn.Module):
    """
    IMPROVED: Asymmetric Loss for multi-label classification
    Better handles class imbalance than Focal Loss
    """
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
    
    def forward(self, logits, targets):
        # Sigmoid probabilities
        probs = torch.sigmoid(logits)
        
        # Asymmetric clipping
        probs_neg = probs.clamp(min=self.clip)
        
        # Positive and negative losses
        pos_loss = targets * torch.log(probs.clamp(min=1e-8))
        neg_loss = (1 - targets) * torch.log((1 - probs_neg).clamp(min=1e-8))
        
        # Asymmetric focusing
        pos_weight = (1 - probs) ** self.gamma_pos
        neg_weight = probs ** self.gamma_neg
        
        loss = -pos_weight * pos_loss - neg_weight * neg_loss
        return loss.mean()

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        bce_loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

class CombinedLoss(nn.Module):
    """IMPROVED: Combined loss for better training"""
    def __init__(self, alpha=0.5):
        super().__init__()
        self.focal = FocalLoss(alpha=0.25, gamma=2.0)
        self.asymmetric = AsymmetricLoss(gamma_neg=4, gamma_pos=1)
        self.alpha = alpha
    
    def forward(self, logits, targets):
        return self.alpha * self.focal(logits, targets) + (1 - self.alpha) * self.asymmetric(logits, targets)

def train_epoch(model, loader, optimizer, criterion, A_hat, device, scheduler=None):
    model.train()
    total_loss = 0
    
    for batch in tqdm(loader, desc="Training"):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(ids, mask, A_hat)
        loss = criterion(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        if scheduler:
            scheduler.step()
        
        total_loss += loss.item()
    
    return total_loss / len(loader)

def evaluate(model, loader, A_hat, device, threshold=0.5):
    model.eval()
    preds, targets = [], []
    
    with torch.no_grad():
        for batch in loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            
            logits = model(ids, mask, A_hat)
            pred = (torch.sigmoid(logits) > threshold).float()
            
            preds.append(pred.cpu().numpy())
            targets.append(lbls.cpu().numpy())
    
    preds = np.concatenate(preds)
    targets = np.concatenate(targets)
    
    tp = np.sum(preds * targets)
    fp = np.sum(preds * (1 - targets))
    fn = np.sum((1 - preds) * targets)
    micro_f1 = 2 * tp / (2 * tp + fp + fn + 1e-10)
    
    tp_per_class = np.sum(preds * targets, axis=0)
    fp_per_class = np.sum(preds * (1 - targets), axis=0)
    fn_per_class = np.sum((1 - preds) * targets, axis=0)
    f1_per_class = 2 * tp_per_class / (2 * tp_per_class + fp_per_class + fn_per_class + 1e-10)
    macro_f1 = np.mean(f1_per_class)
    
    return micro_f1, macro_f1

In [ ]:
# --- Prepare Data ---

id_to_doc = {d['id']: d for d in all_docs}

train_texts = []
train_labels_remapped = {}

for key, labels in combined_train_labels.items():
    if isinstance(key, int):
        if key in id_to_doc:
            doc = id_to_doc[key]
            train_texts.append(doc['text'])
            train_labels_remapped[len(train_texts) - 1] = labels
        elif 0 <= key < len(all_docs):
            doc = all_docs[key]
            train_texts.append(doc['text'])
            train_labels_remapped[len(train_texts) - 1] = labels

print(f"Prepared {len(train_texts)} training samples")

if val_data:
    val_texts = [item['text'] for item in val_data]
    val_labels_remapped = {i: item['labels'] for i, item in enumerate(val_data)}
    print(f"Prepared {len(val_texts)} validation samples")
else:
    from sklearn.model_selection import train_test_split
    train_indices = list(range(len(train_texts)))
    train_idx, val_idx = train_test_split(train_indices, test_size=0.2, random_state=42)
    
    val_texts = [train_texts[i] for i in val_idx]
    val_labels_remapped = {i: train_labels_remapped[val_idx[i]] for i in range(len(val_idx))}
    
    train_texts = [train_texts[i] for i in train_idx]
    new_train_labels = {i: train_labels_remapped[train_idx[i]] for i in range(len(train_idx))}
    train_labels_remapped = new_train_labels
    
    print(f"Split: {len(train_texts)} train, {len(val_texts)} val")

In [ ]:
# --- IMPROVED Training Loop with Curriculum Learning ---

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

train_ds = TextDataset(train_texts, train_labels_remapped, tokenizer, num_classes)
val_ds = TextDataset(val_texts, val_labels_remapped, tokenizer, num_classes)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=32)

# Use improved model
model = HierarchicalGCNClassifier(num_classes, gcn_layers=3).to(device)

# IMPROVED: Use different learning rates for BERT and classifier
bert_params = list(model.bert.parameters())
other_params = [p for n, p in model.named_parameters() if 'bert' not in n]

optimizer = torch.optim.AdamW([
    {'params': bert_params, 'lr': 1e-5},
    {'params': other_params, 'lr': 2e-4}
], weight_decay=0.01)

# Use combined loss
criterion = CombinedLoss(alpha=0.6)

# IMPROVED: Warmup + Cosine Annealing
total_steps = len(train_loader) * 15  # Total training steps
warmup_steps = len(train_loader) * 2  # 2 epochs warmup

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    return max(0.1, 0.5 * (1.0 + np.cos(np.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_f1 = 0
SELF_TRAIN_ROUNDS = 4  # Increased rounds
EPOCHS_PER_ROUND = 5

print("\n=== Starting Improved Training ===")

for round_num in range(SELF_TRAIN_ROUNDS):
    print(f"\n{'='*50}")
    print(f"Round {round_num + 1}/{SELF_TRAIN_ROUNDS}")
    print(f"{'='*50}")
    
    # IMPROVED: Curriculum learning - start with higher threshold, gradually decrease
    confidence_threshold = 0.6 - (round_num * 0.05)  # 0.6 -> 0.55 -> 0.50 -> 0.45
    print(f"Confidence threshold: {confidence_threshold:.2f}")
    
    for epoch in range(EPOCHS_PER_ROUND):
        loss = train_epoch(model, train_loader, optimizer, criterion, A_hat, device)
        micro_f1, macro_f1 = evaluate(model, val_loader, A_hat, device)
        scheduler.step()
        
        print(f"  Epoch {epoch+1}/{EPOCHS_PER_ROUND}: Loss={loss:.4f}, "
              f"Micro-F1={micro_f1:.4f}, Macro-F1={macro_f1:.4f}")
        
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            model_path = os.path.join(MODEL_SAVE_DIR, "best_model_improved.pt")
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'f1': best_f1,
                'round': round_num,
                'epoch': epoch
            }, model_path)
            print(f"    -> Best model saved! F1={best_f1:.4f}")
    
    # IMPROVED: Progressive pseudo-label update with quality filtering
    if round_num < SELF_TRAIN_ROUNDS - 1:
        print("\n  Updating pseudo-labels with quality filtering...")
        model.eval()
        
        inference_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
        new_labels = {}
        empty_fix_count = 0
        confident_update_count = 0
        
        # Collect all predictions and confidence scores
        all_probs = []
        with torch.no_grad():
            for batch in tqdm(inference_loader, desc="Inference"):
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(ids, mask, A_hat)
                probs = torch.sigmoid(logits).cpu()
                all_probs.append(probs)
        
        all_probs = torch.cat(all_probs, dim=0)
        
        # IMPROVED: Use entropy-based quality filtering
        for global_idx in range(len(train_texts)):
            prob_vec = all_probs[global_idx]
            
            # Get original labels (CRITICAL)
            original_labels = train_labels_remapped.get(global_idx, [])
            
            # Calculate prediction entropy (lower = more confident)
            entropy = -torch.sum(prob_vec * torch.log(prob_vec.clamp(min=1e-8)) + 
                                 (1-prob_vec) * torch.log((1-prob_vec).clamp(min=1e-8)))
            entropy = entropy.item() / num_classes
            
            # Get high confidence predictions
            high_conf = (prob_vec > confidence_threshold).nonzero(as_tuple=True)[0].tolist()
            
            # IMPROVED: Quality-based update strategy
            if len(high_conf) >= 2 and entropy < 0.5:  # Low entropy = confident
                # Apply hierarchy constraint
                high_conf = enforce_hierarchy_constraint(high_conf, G)
                
                # Prioritize leaf nodes
                leaf_preds = [c for c in high_conf if c in leaf_nodes]
                
                if len(leaf_preds) >= 1:
                    # Keep at least one leaf prediction
                    selected = leaf_preds[:2]
                    selected = enforce_hierarchy_constraint(selected, G)
                else:
                    selected = high_conf
                
                # Limit to top 3
                if len(selected) > 3:
                    top_probs = [(c, prob_vec[c].item()) for c in selected]
                    top_probs.sort(key=lambda x: x[1], reverse=True)
                    selected = [c for c, _ in top_probs[:3]]
                
                new_labels[global_idx] = selected
                confident_update_count += 1
            
            else:
                # CRITICAL FIX: Keep original labels if no confident predictions
                if len(original_labels) >= 2:
                    new_labels[global_idx] = original_labels
                else:
                    # Last resort: use top-3 predictions even if not confident
                    top_indices = np.argsort(prob_vec.numpy())[-3:].tolist()
                    top_indices = enforce_hierarchy_constraint(top_indices, G)
                    
                    # Ensure 2-3 labels
                    if len(top_indices) > 3:
                        top_probs = [(c, prob_vec[c].item()) for c in top_indices]
                        top_probs.sort(key=lambda x: x[1], reverse=True)
                        top_indices = [c for c, _ in top_probs[:3]]
                    elif len(top_indices) < 2:
                        # Absolute fallback
                        top_indices = np.argsort(prob_vec.numpy())[-2:].tolist()
                    
                    new_labels[global_idx] = top_indices
                    empty_fix_count += 1
        
        # Final safety check: ensure ALL indices have labels
        for idx in range(len(train_texts)):
            if idx not in new_labels or len(new_labels[idx]) == 0:
                if idx in train_labels_remapped and len(train_labels_remapped[idx]) > 0:
                    new_labels[idx] = train_labels_remapped[idx]
                else:
                    # Emergency: assign most common labels
                    new_labels[idx] = [0, 1]  # Root classes as fallback
                empty_fix_count += 1
        
        # Verify label quality
        empty_count = sum(1 for v in new_labels.values() if len(v) == 0)
        avg_labels = np.mean([len(v) for v in new_labels.values()])
        changed = sum(1 for k in new_labels if new_labels[k] != train_labels_remapped.get(k, []))
        leaf_ratio = sum(1 for v in new_labels.values() if any(l in leaf_nodes for l in v)) / len(new_labels)
        
        print(f"  Confident updates: {confident_update_count}")
        print(f"  Empty label fixes: {empty_fix_count}")
        print(f"  Total changed: {changed}/{len(new_labels)}")
        print(f"  Avg labels per doc: {avg_labels:.2f}")
        print(f"  Docs with leaf labels: {leaf_ratio*100:.1f}%")
        
        if empty_count > 0:
            print(f"  ⚠️  WARNING: {empty_count} samples still have empty labels!")
        else:
            print(f"  ✓ All samples have valid labels")
        
        # Update dataset
        train_labels_remapped = new_labels
        train_ds = TextDataset(train_texts, train_labels_remapped, tokenizer, num_classes)
        train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)

print(f"\n=== Training Complete. Best Macro F1: {best_f1:.4f} ===")

In [ ]:
# --- IMPROVED Test Prediction with Dynamic Thresholding ---

def find_optimal_threshold(model, val_loader, A_hat, device):
    """Find optimal threshold using validation set"""
    model.eval()
    all_probs = []
    all_targets = []
    
    with torch.no_grad():
        for batch in val_loader:
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels']
            
            logits = model(ids, mask, A_hat)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            all_probs.append(probs)
            all_targets.append(lbls.numpy())
    
    all_probs = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets)
    
    # Search for optimal threshold
    best_threshold = 0.5
    best_f1 = 0
    
    for threshold in np.arange(0.2, 0.6, 0.02):
        preds = (all_probs > threshold).astype(float)
        tp = np.sum(preds * all_targets)
        fp = np.sum(preds * (1 - all_targets))
        fn = np.sum((1 - preds) * all_targets)
        f1 = 2 * tp / (2 * tp + fp + fn + 1e-10) #micro-f1 기준
        
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
    
    print(f"Optimal threshold: {best_threshold:.2f} (F1: {best_f1:.4f})")
    return best_threshold

def predict_test_improved(model, test_docs, tokenizer, A_hat, device, G, 
                          threshold=0.35, batch_size=32):
    """IMPROVED prediction with hierarchy-aware post-processing"""
    model.eval()
    
    dummy_labels = {i: [] for i in range(len(test_docs))}
    test_texts = [d['text'] for d in test_docs]
    
    test_ds = TextDataset(test_texts, dummy_labels, tokenizer, num_classes)
    test_loader = DataLoader(test_ds, batch_size=batch_size)
    
    all_predictions = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            
            logits = model(ids, mask, A_hat)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            for prob_vec in probs:
                # Get top predictions
                top_indices = np.argsort(prob_vec)[-7:][::-1]  # Get more candidates
                
                # Filter by threshold
                predictions = [idx for idx in top_indices if prob_vec[idx] > threshold]
                
                # IMPROVED: Prioritize leaf nodes
                leaf_preds = [p for p in predictions if p in leaf_nodes]
                non_leaf_preds = [p for p in predictions if p not in leaf_nodes]
                
                if len(leaf_preds) >= 1:
                    # Ensure at least one leaf prediction
                    final_preds = leaf_preds[:2]
                else:
                    final_preds = predictions[:2] if len(predictions) >= 2 else top_indices[:2].tolist()
                
                # Apply hierarchy constraint
                final_preds = enforce_hierarchy_constraint(final_preds, G)
                
                # Limit output
                if len(final_preds) > 3:
                    # Keep leaf nodes and highest probability others
                    leaf_in_final = [p for p in final_preds if p in leaf_nodes]
                    others = [p for p in final_preds if p not in leaf_nodes]
                    others_sorted = sorted(others, key=lambda x: prob_vec[x], reverse=True)
                    
                    final_preds = leaf_in_final[:2] + others_sorted[:max(0, 3-len(leaf_in_final[:2]))]
                    final_preds = final_preds[:3]
                
                all_predictions.append(final_preds)
    
    return all_predictions

# Load best model
checkpoint = torch.load(os.path.join(MODEL_SAVE_DIR, "best_model_improved.pt"))
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model with F1: {checkpoint['f1']:.4f}")

# Find optimal threshold
optimal_threshold = find_optimal_threshold(model, val_loader, A_hat, device)

# Make predictions
predictions = predict_test_improved(model, test_docs, tokenizer, A_hat, device, G, 
                                     threshold=optimal_threshold)

# Save predictions
output_path = os.path.join(MODEL_SAVE_DIR, "predictions_improved.txt")
with open(output_path, 'w') as f:
    for doc, preds in zip(test_docs, predictions):
        pred_str = ' '.join(map(str, sorted(preds)))
        f.write(f"{doc['id']}\t{pred_str}\n")

print(f"\nPredictions saved to {output_path}")
print(f"Avg predictions per document: {np.mean([len(p) for p in predictions]):.2f}")
leaf_coverage = sum(1 for p in predictions if any(l in leaf_nodes for l in p)) / len(predictions)
print(f"Documents with leaf predictions: {leaf_coverage*100:.1f}%")

## Summary of Improvements

### 1. Silver Label Generation
- **Hybrid approach**: TF-IDF + Keyword matching + Label propagation
- **Hierarchy-aware scoring**: Depth-based weighting for specific classes
- **Leaf node prioritization**: Focus on most specific categories

### 2. Model Architecture (GNN-based)
- **Residual GCN layers**: Better gradient flow with 3 layers
- **Mean pooling**: More robust than [CLS] token alone
- **Deeper classifier**: Additional hidden layer with GELU activation
- **Label co-occurrence in adjacency**: Captures statistical dependencies

### 3. Self-Training Strategy
- **Curriculum learning**: Progressive threshold decay (0.6 → 0.45)
- **Entropy-based filtering**: Only update confident predictions
- **Leaf node preservation**: Maintain specific class predictions
- **4 training rounds**: More iterations for refinement

### 4. Loss Function
- **Combined Loss**: Focal Loss + Asymmetric Loss
- **Better class imbalance handling**: Asymmetric loss for multi-label

### 5. Training Optimization
- **Differential learning rates**: 1e-5 for BERT, 2e-4 for classifier
- **Warmup + Cosine annealing**: Better learning rate scheduling

### 6. Test Prediction
- **Dynamic threshold**: Optimized on validation set
- **Hierarchy-aware post-processing**: Prioritize leaf nodes